In [0]:
%run "./02_Silver_Transformation HML"

In [0]:
from pyspark.sql.functions import col, avg, max, count, when, round

# 1. Aggregate the Silver Data into Business-Level Metrics
gold_df = silver_df.groupBy("patient_id").agg(
    round(avg("heart_rate"), 2).alias("avg_heart_rate"),
    round(avg("temperature"), 2).alias("avg_temperature"),
    max("temperature").alias("max_temperature_recorded"),
    count("patient_id").alias("total_readings")
)

# 2. Add a Business Rule: Flag critical patients!
# If a patient's max temperature is over 100.4 F, flag them as "CRITICAL"
gold_df = gold_df.withColumn(
    "patient_status",
    when(col("max_temperature_recorded") >= 100.4, "CRITICAL").otherwise("STABLE")
)

# 3. Display the final Executive Dashboard view!
display(gold_df)

In [0]:
# Save the aggregated data as a permanent Delta Table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("patient_vitals_gold")

print("Gold Table successfully saved to Databricks Hive Metastore!")